# Week 1: Foundations — Cleaning & Exploring the Applicant Dataset

**Name:** Yumnah
**Internship:** NextGenLearners Data Analyst Internship
**Tools:** Python, Pandas, Jupyter Notebook / Google Colab

### Objective
Practice data cleaning and basic exploration with Pandas. I will find and fix missing values, duplicate applicants, inconsistent text and wrong data types in the raw applicant file, explain every decision, and export a clean CSV.

## Step 1: Import Libraries

In [1]:
import pandas as pd
import numpy as np

## Step 2: Load the Raw Data
I read every column as text (`dtype=str`) so that phone numbers keep their leading zero (e.g. `0301...`).

In [2]:
df = pd.read_csv('applicants.csv', dtype=str)
df.head()

,APPLICANT NAmE,email,PhOne,domain applieD,UNIVERsity,application Date,STATUS
0,Noor Baig,noor.baig781@yahoo.com,03619049278,data sci,LUMS,6/4/2026,Selected
1,Hassan Siddiqui,hassan.siddiqui151@yahoo.com,0311-7406078,Web Dev,GIKI,6/11/2026,Selected
2,Faisal Ali,faisal.ali578@yahoo.com,03516568507,webdev,NUST,5/3/2026,Rejected
3,Saad Akhtar,saad.akhtar886@gmail.com,03348878705,Machine Learning,lums,27/04/2026,NaN
4,Komal Mirza,komal.mirza578@gmail.com,03507670232,UI/UX Design,lums,12/4/2026,Rejected


## Step 3: First Look — Understand Before Cleaning

In [3]:
df.shape

(500, 7)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   APPLICANT NAmE    488 non-null    object
 1   email             469 non-null    object
 2   PhOne             489 non-null    object
 3   domain applieD    500 non-null    object
 4   UNIVERsity        474 non-null    object
 5   application Date  500 non-null    object
 6   STATUS            472 non-null    object
dtypes: object(7)
memory usage: 27.5+ KB


In [5]:
df.isnull().sum()

APPLICANT NAmE      12
email               31
PhOne               11
domain applieD       0
UNIVERsity          26
application Date     0
STATUS              28
dtype: int64

In [6]:
df.duplicated().sum()

np.int64(0)

In [7]:
df.columns

Index(['APPLICANT NAmE', 'email', 'PhOne', 'domain applieD', 'UNIVERsity',
       'application Date', 'STATUS'],
      dtype='object')

### What I observed
- The dataset has **500 rows and 7 columns**.
- The column names are messy (`APPLICANT NAmE`, `PhOne`, `UNIVERsity`...), so I will rename them.
- Missing values: Applicant Name 12, Email 31, Phone 11, University 26, Status 28.
- There are **0 exact duplicate rows**, but near-duplicates may still exist (same person, different formatting).
- All columns are text. `Application Date` is plain text, not a real date.

### Rename the columns to clean, consistent names

In [8]:
df.columns = ['Applicant Name', 'Email', 'Phone', 'Domain Applied', 'University', 'Application Date', 'Status']
df.columns

Index(['Applicant Name', 'Email', 'Phone', 'Domain Applied', 'University',
       'Application Date', 'Status'],
      dtype='object')

## Step 4: Handle Missing Values (Column by Column)
There is no single correct rule, so I decide for each column and explain why.

**Applicant Name:** I drop these rows. An application without a name cannot be identified or contacted.

In [9]:
df = df.dropna(subset=['Applicant Name'])
df.shape

(488, 7)

**Email:** I fill missing emails with a placeholder instead of dropping the rows, so the rest of the application can still be used for analysis. In Step 5 I make sure these placeholders are not treated as duplicates.

In [10]:
placeholder = 'not_provided@example.com'
df['Email'] = df['Email'].fillna(placeholder)

**University:** Not essential, so I fill with `Not Specified` instead of dropping the whole row.

In [11]:
df['University'] = df['University'].fillna('Not Specified')

**Status:** A missing status most likely means the application is not processed yet, so I fill with `Under Review`.

In [12]:
df['Status'] = df['Status'].fillna('Under Review')

**Phone:** I leave missing phone numbers as missing. A phone number cannot be guessed, and placeholder text inside a phone column would make it messy.

In [13]:
df.isnull().sum()

Applicant Name       0
Email                0
Phone               11
Domain Applied       0
University           0
Application Date     0
Status               0
dtype: int64

## Step 5: Remove Duplicate Applicants
I check exact duplicates first. Then I check near-duplicates by **Email**, because email is a more reliable unique ID than name (two different people can have the same name, but not the same email).

In [15]:
df = df.drop_duplicates()
df.duplicated().sum()

np.int64(0)

Before comparing emails I remove extra spaces and make them lowercase, so `ALI@Gmail.com ` and `ali@gmail.com` match.

In [16]:
df['Email'] = df['Email'].str.strip().str.lower()

Let me look at the repeated emails first (placeholders are ignored).

In [17]:
repeated = df[(df['Email'] != placeholder) & df['Email'].duplicated(keep=False)]
repeated.sort_values('Email')

,Applicant Name,Email,Phone,Domain Applied,University,Application Date,Status
160,Ayesha Ali,ayesha.ali373@yahoo.com,03534672762,WEB DEVELOPMENT,Comsats,19-Apr-26,Rejected
385,AYESHA ALI,ayesha.ali373@yahoo.com,03534672762,WEB DEVELOPMENT,Comsats,19-Apr-26,Rejected
417,Ayesha Raza,ayesha.raza354@hotmail.com,0341-0007792,data science,Nust,1/21/2026,Rejected
492,AYESHA RAZA,ayesha.raza354@hotmail.com,0341 0007792,data science,Nust,1/21/2026,Rejected
1,Hassan Siddiqui,hassan.siddiqui151@yahoo.com,0311-7406078,Web Dev,GIKI,6/11/2026,Selected
348,HASSAN SIDDIQUI,hassan.siddiqui151@yahoo.com,0311 7406078,Web Dev,GIKI,6/11/2026,Selected


These rows have the same email, phone, domain and date, with only name capitalization or phone format different, so they are duplicate applications. I keep the first one.

In [18]:
df = df[~((df['Email'] != placeholder) & df['Email'].duplicated())].copy()
df.shape

(485, 7)

In [19]:
df[df['Email'] != placeholder]['Email'].duplicated().sum()

np.int64(0)

I found and removed **3 near-duplicate applications** (same email) in addition to 0 exact duplicates.

## Step 6: Standardize Text Fields
The same value can appear in many forms (`web dev`, `Web Development`, `WEB DEV`). Python sees them as different, so I standardize them.

### 6.1 Applicant Name

In [20]:
df['Applicant Name'] = df['Applicant Name'].str.strip().str.title()
df['Applicant Name'].head()

0          Noor Baig
1    Hassan Siddiqui
2         Faisal Ali
3        Saad Akhtar
4        Komal Mirza
Name: Applicant Name, dtype: object

### 6.2 Domain Applied
First I strip spaces and make everything lowercase, then I look at all the spellings.

In [21]:
df['Domain Applied'] = df['Domain Applied'].str.strip().str.lower()
df['Domain Applied'].value_counts()

Domain Applied
machine learning     76
data science         54
digital marketing    46
digital mktg         46
web dev              45
ui/ux                35
web development      34
ui ux design         30
ml                   29
uiux                 23
webdev               20
data  science        16
ui/ux design         15
data sci             11
ds                    5
Name: count, dtype: int64

Now I build a mapping dictionary from every spelling to the 5 correct domain names.

In [22]:
domain_mapping = {'data science': 'Data Science', 'data  science': 'Data Science', 'data sci': 'Data Science', 'ds': 'Data Science'}
domain_mapping.update({'web development': 'Web Development', 'web dev': 'Web Development', 'webdev': 'Web Development'})
domain_mapping.update({'machine learning': 'Machine Learning', 'ml': 'Machine Learning'})

In [23]:
domain_mapping.update({'ui/ux design': 'UI/UX Design', 'ui ux design': 'UI/UX Design', 'ui/ux': 'UI/UX Design', 'uiux': 'UI/UX Design'})
domain_mapping.update({'digital marketing': 'Digital Marketing', 'digital mktg': 'Digital Marketing'})

In [24]:
df['Domain Applied'] = df['Domain Applied'].map(domain_mapping)
df['Domain Applied'].unique()

array(['Data Science', 'Web Development', 'Machine Learning',
       'UI/UX Design', 'Digital Marketing'], dtype=object)

In [25]:
df['Domain Applied'].isnull().sum()

np.int64(0)

I re-checked with `.unique()`: exactly 5 domains remain and there are 0 `NaN` values, so no spelling was missed. (If `NaN` ever appears, add the missing spelling to the dictionary and run again.)

### 6.3 University
I look at the values first. Abbreviations like `KU`, `AU`, `PU`, `IBA` and `UET` are mapped to the full university name. I treat `UET` and `UET Lahore` as the same university.

In [26]:
uni_clean = df['University'].str.strip()
uni_key = uni_clean.str.lower()
uni_clean.value_counts()

University
KU                          25
Not Specified               24
AU                          24
uet lahore                  22
IBA Karachi                 20
LUMS                        20
GIKI                        19
IBA                         19
lums                        19
UET Lahore                  17
UET                         17
Air University              16
fast-nuces                  16
nust                        15
air uni                     14
Lums                        13
iba karachi                 13
Nust                        13
University of the Punjab    12
University of Karachi       12
giki                        12
comsats uni                 11
Comsats                     11
COMSATS University          11
Karachi Univ                11
Punjab University           10
FAST NUCES                  10
FAST                        10
NUST                        10
Fast Nuces                  10
PU                          10
Giki                        

In [27]:
uni_mapping = {'lums': 'LUMS', 'nust': 'NUST', 'giki': 'GIKI', 'not specified': 'Not Specified'}
uni_mapping.update({'iba': 'IBA Karachi', 'iba karachi': 'IBA Karachi'})
uni_mapping.update({'uet': 'UET Lahore', 'uet lahore': 'UET Lahore'})

In [28]:
uni_mapping.update({'air university': 'Air University', 'air uni': 'Air University', 'au': 'Air University'})
uni_mapping.update({'fast': 'FAST-NUCES', 'fast nuces': 'FAST-NUCES', 'fast-nuces': 'FAST-NUCES'})
uni_mapping.update({'comsats': 'COMSATS University', 'comsats university': 'COMSATS University', 'comsats uni': 'COMSATS University'})

In [29]:
uni_mapping.update({'university of the punjab': 'University of the Punjab', 'punjab university': 'University of the Punjab', 'univ of punjab': 'University of the Punjab', 'pu': 'University of the Punjab'})
uni_mapping.update({'university of karachi': 'University of Karachi', 'karachi univ': 'University of Karachi', 'ku': 'University of Karachi'})

In [30]:
df['University'] = uni_key.map(uni_mapping).fillna(uni_clean)
df['University'].value_counts()

University
UET Lahore                  56
Air University              54
LUMS                        52
IBA Karachi                 52
University of Karachi       48
FAST-NUCES                  46
COMSATS University          40
NUST                        38
GIKI                        38
University of the Punjab    37
Not Specified               24
Name: count, dtype: int64

### 6.4 Status
Make `selected`, `Selected`, `SELECTED` and ` Selected` all the same.

In [31]:
df['Status'] = df['Status'].str.strip().str.title()
df['Status'].unique()

array(['Selected', 'Rejected', 'Under Review'], dtype=object)

## Step 7: Fix Data Types (Dates and Phone Numbers)

### 7.1 Application Date
Let me look at the different date formats.

In [32]:
df['Application Date'] = df['Application Date'].str.strip()
df['Application Date'].sample(15, random_state=1)

32      29-Mar-26
250    28/03/2026
191    20/03/2026
134     25-Jun-26
92     17/06/2026
164     6/27/2026
408    24/04/2026
121     6/12/2026
316     12-Feb-26
275     3/19/2026
161      4/3/2026
35       5/2/2026
68      2/11/2026
64      4/12/2026
334     21-May-26
Name: Application Date, dtype: object

I found **3 formats**, plus values that are not dates:
- `6/4/2026` — month/day/year, no leading zeros
- `27/04/2026` — day/month/year, with leading zeros (the first number is above 12, so it must be the day)
- `4-Apr-26` — day-month name-year
- `TBD`, `not available` and `32/13/2026` (an impossible date) are not real dates

I convert each format separately with `errors='coerce'` (unreadable values become `NaT`, Pandas' missing date), then combine the results with `fillna`.

In [33]:
raw_dates = df['Application Date']
d1 = pd.to_datetime(raw_dates, format='%m/%d/%Y', errors='coerce')
d2 = pd.to_datetime(raw_dates, format='%d/%m/%Y', errors='coerce')

In [34]:
d3 = pd.to_datetime(raw_dates, format='%d-%b-%y', errors='coerce')
df['Application Date'] = d1.fillna(d2).fillna(d3)

In [35]:
df['Application Date'].isnull().sum()

np.int64(7)

In [36]:
raw_dates[df['Application Date'].isnull()].unique()

array(['32/13/2026', 'TBD', 'not available'], dtype=object)

7 dates could not be understood (`TBD`, `not available`, `32/13/2026`), so they are `NaT`. I **keep** these rows because the rest of the application is still valid, and I do not replace them with text, because that would turn the column back into text.

**Note about ambiguous dates:** For slash dates without leading zeros (like `6/4/2026`) I assumed month/day/year, because every date that can be checked (second number above 12) follows that order. Most dates fall between January and June 2026, but **24 dates** became July–December, so a few of them might actually be day/month swapped. I cannot confirm this without the original source, so I kept them as parsed and mention it here.

In [37]:
df[df['Application Date'] > '2026-06-30'][['Applicant Name', 'Application Date']].head()

,Applicant Name,Application Date
4,Komal Mirza,2026-12-04
5,Ayesha Shah,2026-12-01
23,Talha Javed,2026-09-04
36,Ali Aslam,2026-08-02
57,Ahmed Malik,2026-10-03


### 7.2 Phone Numbers
I remove dashes and spaces but keep the phone as **text**, so leading zeros are not lost.

In [38]:
df['Phone'] = df['Phone'].str.replace('-', '', regex=False).str.replace(' ', '', regex=False)
df['Phone'].head()

0    03619049278
1    03117406078
2    03516568507
3    03348878705
4    03507670232
Name: Phone, dtype: object

In [39]:
df['Phone'].str.len().value_counts()

Phone
11.0    474
Name: count, dtype: int64

All phone numbers now have the same 11-digit format.

### Verify the data types

In [40]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 485 entries, 0 to 499
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Applicant Name    485 non-null    object        
 1   Email             485 non-null    object        
 2   Phone             474 non-null    object        
 3   Domain Applied    485 non-null    object        
 4   University        485 non-null    object        
 5   Application Date  478 non-null    datetime64[ns]
 6   Status            485 non-null    object        
dtypes: datetime64[ns](1), object(6)
memory usage: 30.3+ KB


## Step 8: Final Checks

In [41]:
df = df.reset_index(drop=True)
df.shape

(485, 7)

In [42]:
df.isnull().sum()

Applicant Name       0
Email                0
Phone               11
Domain Applied       0
University           0
Application Date     7
Status               0
dtype: int64

In [43]:
df.duplicated().sum()

np.int64(0)

In [44]:
print(df['Domain Applied'].nunique())
df['Domain Applied'].unique()

5


array(['Data Science', 'Web Development', 'Machine Learning',
       'UI/UX Design', 'Digital Marketing'], dtype=object)

In [45]:
df['Status'].unique()

array(['Selected', 'Rejected', 'Under Review'], dtype=object)

## Step 9: Data Quality Summary

**Starting dataset:** 500 rows, 7 columns

**Issues found:**
- Messy column names (mixed capital and small letters)
- Missing values: Applicant Name 12, Email 31, Phone 11, University 26, Status 28
- 0 exact duplicate rows, but 3 duplicate applications found by matching Email (different name capitalization / phone format)
- Domain Applied had 30 different spellings for only 5 real domains; University had 33 spellings and Status had 12 for the same few real values
- Application Date was stored as text in 3 different formats, with some invalid values (`TBD`, `not available`, `32/13/2026`)
- Phone numbers had inconsistent dash/space formatting; names and emails had extra spaces and inconsistent capitalization

**Actions taken:**
- Renamed the columns
- Dropped 12 rows with a missing Applicant Name (not usable without identification)
- Filled missing Email with a placeholder, University with "Not Specified" and Status with "Under Review"; kept missing Phone as missing
- Removed 3 duplicate rows by matching Email (placeholder emails were excluded from this check)
- Standardized names, emails, Domain Applied (5 categories), University (10 universities + "Not Specified") and Status (3 values)
- Converted Application Date to a real datetime type; 7 unreadable dates were kept as NaT
- Cleaned Phone formatting (kept as text, 11 digits)

**Final dataset:** 485 rows, 7 columns, no missing Applicant Name, Email, University or Status; 11 missing Phone and 7 missing Application Date are kept on purpose; no duplicate applicants; consistent formatting throughout.

**Limitation:** 24 dates fall in July–December 2026 and may be day/month swapped; this should be verified with the original source.

## Step 10: Export the Cleaned Dataset

In [46]:
df.to_csv('applicants_cleaned.csv', index=False)

Saved applicants_cleaned.csv
